# Workshop: Streaming & Auto Loader

> *"Set up Auto Loader for streaming JSON ingestion into the Bronze layer with exactly-once guarantees."*

**Learning objective:** Ingest files incrementally into Bronze — prove COPY INTO idempotency, then build Auto Loader (`cloudFiles`) streams with checkpoints, `trigger(availableNow=True)`, metadata columns, and schema-rescue handling.

**Expected duration:** ~45 minutes

## Setup

In [0]:
%run ../setup/00_setup

In [0]:
# Prepare landing zone and checkpoint paths
landing_path = f"{DATASET_PATH}/orders/stream"
lab_base = f"{DATASET_PATH}/lab05_work"     # checkpoints, schemas and the Auto Loader landing zone (UC Volume)
checkpoint_path = f"{lab_base}/checkpoint"
schema_path = f"{lab_base}/schema"
target_table = f"{CATALOG}.{BRONZE_SCHEMA}.orders_stream"

# Clean up from previous runs
spark.sql(f"DROP TABLE IF EXISTS {target_table}")
dbutils.fs.rm(checkpoint_path, True)
dbutils.fs.rm(schema_path, True)

print(f"Landing path: {landing_path}")
print(f"Target table: {target_table}")
print(f"Files available: {[f.name for f in dbutils.fs.ls(landing_path)]}")

## Task 1: COPY INTO — Batch Load + Idempotency Proof

> The demo (`05 — Incremental Ingestion`) walked COPY INTO step by step, so here it is **one compact task**: load the landing zone once, then re-run the exact same command and prove that **zero** new rows are inserted.

**What you need to do:**
1. Write a `COPY INTO` statement loading the JSON files from `landing_path` into the (provided) target table
2. Run the **exact same statement again** and compare row counts — idempotency means no duplicates

**Guidance — Task 1**

`COPY INTO` is a SQL command that loads files from cloud storage into a Delta table.

**Syntax:**
```sql
COPY INTO target_table
FROM '/path/to/files'
FILEFORMAT = JSON
FORMAT_OPTIONS ('mergeSchema' = 'true')
```

Key properties:
- **Idempotent** — Databricks records processed files in an internal state table and silently skips them on re-runs; the row count must not change
- **Batch** — loads all matching files in one operation; best for up to thousands of files
- `FORMAT_OPTIONS ('mergeSchema' = 'true')` — **reader level**: infer and merge the schema across all source files
- `COPY_OPTIONS ('mergeSchema' = 'true')` — **table level**: evolve the target Delta table when incoming data brings new columns (not needed here — the target already declares every column)
- `COPY_OPTIONS ('force' = 'true')` — disables idempotency and reloads every file

This is fundamentally different from `INSERT INTO ... SELECT *`, which would duplicate every row on re-run.

**Things to think about**
- How does Databricks know which files have already been loaded?
- How is idempotency useful in a production job scheduled every hour?

In [ ]:
# First create the target table (provided)
spark.sql(f"""
    CREATE OR REPLACE TABLE {target_table}
    (order_id STRING, customer_id STRING, product_id STRING, store_id STRING,
     order_datetime STRING, quantity BIGINT, unit_price DOUBLE,
     discount_percent BIGINT, total_amount DOUBLE, payment_method STRING)
""")

spark.sql(f"""
    COPY INTO {target_table}
    FROM '{landing_path}'
    FILEFORMAT = JSON
    FORMAT_OPTIONS ('mergeSchema' = 'true')
""")

count_after_copy = spark.table(target_table).count()
print(f"Rows after COPY INTO: {count_after_copy}")

# Re-run the EXACT same COPY INTO — idempotency check
spark.sql(f"""
    COPY INTO {target_table}
    FROM '{landing_path}'
    FILEFORMAT = JSON
    FORMAT_OPTIONS ('mergeSchema' = 'true')
""")

count_after_rerun = spark.table(target_table).count()
print(f"Rows after re-run  : {count_after_rerun} (new rows: {count_after_rerun - count_after_copy})")

In [ ]:
# -- Validation --
assert count_after_copy > 0, "COPY INTO should have loaded data"
assert count_after_rerun == count_after_copy, "COPY INTO should be idempotent - no new rows on re-run!"
print(f"Task 1 OK: {count_after_copy} rows loaded; re-run added 0 rows (idempotent)")

## Task 2: Auto Loader - Configure Stream

Set up Auto Loader (`cloudFiles`) to read JSON files from the lab's landing zone `al_landing` (prepared by the reset cell below).

Key options:
- `cloudFiles.format` = json
- `cloudFiles.schemaLocation` = path for inferred schema
- `cloudFiles.inferColumnTypes` = true

**Guidance — Task 2**

Auto Loader uses the `cloudFiles` Structured Streaming source. It tracks processed files using a **checkpoint directory** and stores the inferred schema in a **schema location**.

**Syntax:**
```python
df_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")           # source file format
        .option("cloudFiles.schemaLocation", path)     # where to persist the inferred schema
        .option("cloudFiles.inferColumnTypes", "true") # infer correct types (not just STRING)
        .load(al_landing)
)
```

| Option | Purpose |
|--------|---------|
| `cloudFiles.format` | File format of source files (json, csv, parquet, ...) |
| `cloudFiles.schemaLocation` | Path where the inferred schema is stored between runs |
| `cloudFiles.inferColumnTypes` | Detect proper column types instead of treating all as STRING |

After configuration, verify `df_stream.isStreaming == True` — the DataFrame is a lazy stream, not a batch read.

**Things to think about**
- Why does Auto Loader need a `schemaLocation` while COPY INTO does not?
- What does `inferColumnTypes=true` change compared to the default behaviour?

In [0]:
# Reset target for Auto Loader test
al_target = f"{CATALOG}.{BRONZE_SCHEMA}.orders_autoloader"
spark.sql(f"DROP TABLE IF EXISTS {al_target}")
dbutils.fs.rm(checkpoint_path, True)
dbutils.fs.rm(schema_path, True)

# Private landing zone for Tasks 2-4: all source files except the last one,
# which "arrives" later in Task 4
al_landing = f"{lab_base}/landing"
dbutils.fs.rm(al_landing, True)
stream_files = sorted(f.path for f in dbutils.fs.ls(landing_path) if f.name.endswith(".json"))
for p in stream_files[:-1]:
    dbutils.fs.cp(p, f"{al_landing}/{p.split('/')[-1]}")
late_file = stream_files[-1]
print(f"Auto Loader landing zone: {al_landing} ({len(stream_files) - 1} files; 1 more arrives in Task 4)")

In [0]:
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(al_landing)
)

In [0]:
# -- Validation --
assert df_stream.isStreaming, "Should be a streaming DataFrame"
print(f"Task 2 OK: Streaming DataFrame configured with schema: {df_stream.schema.fieldNames()}")

## Task 3: Write Stream with trigger(availableNow=True)

Write the stream to a Delta table using `trigger(availableNow=True)`.

This processes all available files and stops automatically.

**Guidance — Task 3**

Use `.writeStream` to persist the streaming DataFrame to a Delta table.

**Syntax:**
```python
query = (
    df_stream
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(table_name)
)
query.awaitTermination()
```

| Option | Value | Purpose |
|--------|-------|---------|
| `outputMode` | `"append"` | Add only new rows — correct for ingestion pipelines |
| `checkpointLocation` | path | Records stream progress for exactly-once delivery |
| `trigger(availableNow=True)` | — | Process all pending files and stop automatically |

`awaitTermination()` blocks notebook execution until the stream finishes. Without it, the cell returns immediately while the stream runs in the background.

**Things to think about**
- What would happen if you deleted the checkpoint directory and re-ran the stream?
- What is the difference between `trigger(availableNow=True)` and `trigger(processingTime='1 minute')` (classic compute only — serverless supports `availableNow`)?

In [0]:
query = (
    df_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(al_target)
)

query.awaitTermination()
print(f"Stream completed. Rows loaded: {spark.table(al_target).count()}")

In [0]:
# -- Validation --
al_count = spark.table(al_target).count()
assert al_count > 0, "Auto Loader should have loaded data"
print(f"Task 3 OK: {al_count} rows loaded via Auto Loader")

## Task 4: Incremental Processing — A New File Arrives

Package the stream from Tasks 2–3 into a reusable function and prove the checkpoint works in both directions: a run with **no new files adds 0 rows**, and a run after a new file lands adds **exactly that file's rows**.

**What you need to do:**
1. Complete `run_ingestion()` — the `readStream` from Task 2 plus the `writeStream` from Task 3, with the **same** `schema_path` and `checkpoint_path`, waiting until the run finishes
2. Run the provided cell: Run A (nothing new) → a late file is copied into `al_landing` → Run B

**Guidance — Task 4**

A scheduled ingestion job calls the same code every time — so wrap it in a function instead of copy-pasting it:

```python
def run_ingestion():
    before = spark.table(al_target).count()
    df = (spark.readStream.format("cloudFiles")
            # ... same options as Task 2 ...
            .load(al_landing))
    (df.writeStream
        # ... same options as Task 3 (format, outputMode, checkpointLocation, trigger) ...
        .toTable(al_target)
        .awaitTermination())
    return spark.table(al_target).count() - before
```

The checkpoint records which files have already been processed:
- **Run A** — every file in `al_landing` is already in the checkpoint → **0** new rows
- A new file lands in the landing zone
- **Run B** — Auto Loader discovers only the new file → new rows == rows in that file

Do **not** reset `checkpoint_path` or `schema_path` between runs — that would reprocess everything. If you need to repeat this task, start again from the Task 2 reset cell (it rebuilds the landing zone).

**Things to think about**
- What would happen if you cleared the checkpoint directory before Run B?
- How does Auto Loader's checkpoint differ from COPY INTO's file tracking?

In [0]:
def run_ingestion():
    """One scheduled incremental run over al_landing. Returns the number of rows added."""
    before = spark.table(al_target).count()
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .load(al_landing)
    )
    (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(al_target)
        .awaitTermination()
    )
    return spark.table(al_target).count() - before

# Run A — nothing new in the landing zone (provided)
new_rows_no_files = run_ingestion()
print(f"Run A (no new files): +{new_rows_no_files} rows")

# A new file arrives (provided) — re-running this cell? Start again from the Task 2 reset cell.
dbutils.fs.cp(late_file, f"{al_landing}/{late_file.split('/')[-1]}")
late_file_rows = spark.read.json(late_file).count()

# Run B — only the new file should be processed (provided)
new_rows_late_file = run_ingestion()
al_count2 = spark.table(al_target).count()
print(f"Run B (1 new file)  : +{new_rows_late_file} rows (the new file has {late_file_rows} rows)")

In [0]:
# -- Validation --
assert new_rows_no_files == 0, f"Run A should add 0 rows (checkpoint), but added {new_rows_no_files}"
assert new_rows_late_file == late_file_rows, \
    f"Run B should add exactly the new file's {late_file_rows} rows, but added {new_rows_late_file}"
assert al_count2 == al_count + late_file_rows, "Total rows should be the Task 3 count plus the new file"
print(f"Task 4 OK: Run A +0 rows, Run B +{new_rows_late_file} rows — only the new file was processed")

## Task 5: Metadata Columns

Add metadata columns to streaming data: `_processing_time` (processing timestamp) and `_source_file` (source file path from `_metadata`).

These columns are essential in production pipelines for debugging and auditing.

**TODO:** Fill in `current_timestamp()` and `_metadata.file_path`.

**Guidance — Task 5**

Every file-based Spark source exposes a hidden `_metadata` struct column with file-level information. Add it using `.withColumn()` after `.load()`.

**Pattern:**
```python
from pyspark.sql.functions import current_timestamp, col

df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", path)
        .load(landing_path)
        .withColumn("_processing_time", current_timestamp())
        .withColumn("_source_file",     col("_metadata.file_path"))
)
```

**Available `_metadata` fields:**

| Expression | Type | Content |
|-----------|------|---------|
| `_metadata.file_path` | STRING | Full path to the source file |
| `_metadata.file_name` | STRING | Filename only |
| `_metadata.file_size` | LONG | File size in bytes |
| `_metadata.file_modification_time` | TIMESTAMP | Last modified time |

`current_timestamp()` and `col` are already imported in the cell — you do not need to add them again.

**Things to think about**
- Why is storing `_processing_time` in a Bronze table useful for debugging?
- How would you use `_source_file` to trace back a bad row to its origin file?

In [0]:
from pyspark.sql.functions import current_timestamp, col

metadata_target = f"{CATALOG}.{BRONZE_SCHEMA}.orders_with_metadata"
metadata_checkpoint = f"{lab_base}/checkpoint_metadata"
metadata_schema = f"{lab_base}/schema_metadata"

spark.sql(f"DROP TABLE IF EXISTS {metadata_target}")
dbutils.fs.rm(metadata_checkpoint, True)
dbutils.fs.rm(metadata_schema, True)

df_with_metadata = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", metadata_schema)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing_path)
    .withColumn("_processing_time", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

(
    df_with_metadata
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", metadata_checkpoint)
    .trigger(availableNow=True)
    .toTable(metadata_target)
    .awaitTermination()
)

print(f"Written to: {metadata_target}")
display(spark.table(metadata_target).limit(5))

In [0]:
# -- Validation --
meta_cols = spark.table(metadata_target).columns
assert "_processing_time" in meta_cols, "Missing '_processing_time' — did you use current_timestamp()?"
assert "_source_file" in meta_cols, "Missing '_source_file' — did you use _metadata.file_path?"
print(f"Task 5 OK: Metadata columns added — {meta_cols}")

## Task 6: Schema Evolution — Rescued Data

Configure Auto Loader with a partial schema (only `order_id` + `customer_id`). Set `schemaEvolutionMode` to `"rescue"` so that extra columns land in `_rescued_data`.

**TODO:** Fill in the `schemaEvolutionMode` value.

**Guidance — Task 6**

When Auto Loader reads a file with **more columns than the declared schema**, you control what happens via `cloudFiles.schemaEvolutionMode`.

| Mode | Behaviour |
|------|-----------|
| `addNewColumns` (default) | Adds new columns to the schema automatically |
| `rescue` | Captures unknown columns as JSON in `_rescued_data` |
| `none` | Silently drops unknown columns |
| `failOnNewColumns` | Raises an error if the schema changes |

For this task, use `"rescue"` — extra columns are serialised as a JSON string in `_rescued_data` without modifying the schema.

**Syntax:**
```python
df_rescued = (
    spark.readStream
        .format("cloudFiles")
        .schema(partial_schema)                                      # only order_id + customer_id
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", rescue_schema)
        .option("cloudFiles.schemaEvolutionMode", "rescue")          # extra columns go to _rescued_data
        .load(landing_path)
)
```

Because `partial_schema` only declares `order_id` and `customer_id`, all other columns from the JSON (quantity, total_amount, etc.) are captured as JSON in the `_rescued_data` column.

**Things to think about**
- When would you prefer `rescue` over `addNewColumns`?
- How would you extract a specific field from `_rescued_data` in a downstream Silver transformation?

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

rescue_target = f"{CATALOG}.{BRONZE_SCHEMA}.orders_rescued"
rescue_checkpoint = f"{lab_base}/checkpoint_rescue"
rescue_schema = f"{lab_base}/schema_rescue"

spark.sql(f"DROP TABLE IF EXISTS {rescue_target}")
dbutils.fs.rm(rescue_checkpoint, True)
dbutils.fs.rm(rescue_schema, True)

partial_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
])

df_rescued = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", rescue_schema)
    .schema(partial_schema)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .load(landing_path)
)

(
    df_rescued
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", rescue_checkpoint)
    .trigger(availableNow=True)
    .toTable(rescue_target)
    .awaitTermination()
)

print(f"Written to: {rescue_target}")
display(spark.table(rescue_target).limit(5))

In [0]:
# -- Validation --
rescue_cols = spark.table(rescue_target).columns
assert "_rescued_data" in rescue_cols, "Missing '_rescued_data' column — did you set schemaEvolutionMode to 'rescue'?"
rescue_count = spark.table(rescue_target).filter("_rescued_data IS NOT NULL").count()
assert rescue_count > 0, "Expected rescued data for columns not in partial schema"
print(f"Task 6 OK: {rescue_count} rows with rescued data (extra columns captured in _rescued_data)")

## Cleanup

In [0]:
# Stop any active streams
for s in spark.streams.active:
    s.stop()

# Drop lab tables
for t in [target_table, al_target,
          f"{CATALOG}.{BRONZE_SCHEMA}.orders_with_metadata",
          f"{CATALOG}.{BRONZE_SCHEMA}.orders_rescued"]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

# Clean checkpoints, schemas and the lab landing zone (all under lab_base)
dbutils.fs.rm(lab_base, True)

print("Lab cleanup complete")

## Lab Complete

You have:
- Used COPY INTO for idempotent batch loading
- Configured Auto Loader (`cloudFiles`) for streaming ingestion
- Used `trigger(availableNow=True)` for incremental processing
- Verified checkpoint-based exactly-once guarantees: 0 rows with no new files, only the new file's rows after it arrived
- Added metadata columns (`_processing_time`, `_source_file`) to streaming writes
- Used rescued data column for schema evolution handling

| Feature | COPY INTO | Auto Loader |
|---------|-----------|-------------|
| Format | SQL command | `readStream` / `writeStream` |
| Scalability | Thousands of files | Millions of files |
| Schema evolution | Manual | Automatic (rescue column) |
| File tracking | SQL state | Checkpoint directory |
| Use case | Simple batch | File-based streaming |

> **Exam Tip:** Auto Loader uses `cloudFiles` format. COPY INTO is simpler but Auto Loader scales better (directory listing by default; file notification mode with file events for millions of files). For Stream-Static Joins and Change Data Feed — see the Bonus notebook.


← [05 — Incremental Ingestion](../day2/demo/05_incremental_ingestion.ipynb) | **[ README](../../README.md)** | [06 — Medallion Architecture →](../day2/demo/06_medallion_architecture.ipynb)